# 🧭 GIADA Task 7c — boundary-state semantics
Rianalisi forense delle tracce 7b già salvate: nessun nuovo teacher e nessun retraining. Lo stato autentico al bordo richiede V prima e gate dopo. Non è una conferma fresh.

In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_task_7c');WORK.mkdir(parents=True,exist_ok=True)
GIADA_REPO=WORK/'giada';TEACHER_REPO=WORK/'neuron_as_deep_net'
if not GIADA_REPO.is_dir():subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
if not TEACHER_REPO.is_dir():subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip();print({'revision':REVISION})


In [ ]:
import torch
assert torch.cuda.is_available(),'Task 7c richiede GPU CUDA per gli stessi modelli congelati della Task 7b.'
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]:del sys.modules[name]
from src.giada_teacher.voltage_path_stress import EXPECTED_TASK5_ARCHIVE_SHA256,EXPECTED_TASK5_REPORT_SHA256,verified_task5_root
from src.giada_teacher.cahva_boundary_semantics_reassessment import verified_task7b_trace_bytes
plan=json.loads((GIADA_REPO/'experiments/teacher_cahva_boundary_semantics_reassessment_plan_v1.json').read_text())
print({'study_type':plan['study_type'],'cuda':torch.cuda.get_device_name(0),'expected_episodes':plan['checks']['episode_count']})


In [ ]:
def file_sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()
INPUT_ROOT=Path('/kaggle/input');override5=os.environ.get('GIADA_TASK5_ARTIFACT');override7b=os.environ.get('GIADA_TASK7B_ARTIFACT')
candidates5=[Path(override5).expanduser()] if override5 else []
candidates7b=[Path(override7b).expanduser()] if override7b else []
if INPUT_ROOT.is_dir():
    candidates5+=list(INPUT_ROOT.rglob('giada_primitive_scaling_laws.zip'))+list(INPUT_ROOT.rglob('archive.zip'))
    candidates5+=[p.parent for p in INPUT_ROOT.rglob('final_report.json') if (p.parent/'frozen_scaling_checkpoints.pt').is_file()]
    candidates7b+=list(INPUT_ROOT.rglob('giada_cahva_active_closed_loop_microcanary.zip'))+list(INPUT_ROOT.rglob('archive.zip'))
    candidates7b+=[p.parent for p in INPUT_ROOT.rglob('final_report.json') if (p.parent/'trajectories.npz').is_file() and 'giada_cahva_active_closed_loop_microcanary' in str(p.parent)]
def exact5(path):
    try:return file_sha(path)==EXPECTED_TASK5_ARCHIVE_SHA256 if path.is_file() else file_sha(path/'final_report.json')==EXPECTED_TASK5_REPORT_SHA256
    except Exception:return False
TASK5_SOURCE=next((p.resolve() for p in candidates5 if p.exists() and exact5(p)),None)
assert TASK5_SOURCE is not None,'Aggiungi giada_primitive_scaling_laws agli Input oppure imposta GIADA_TASK5_ARTIFACT.'
TASK5_ROOT=verified_task5_root(TASK5_SOURCE,WORK/'.verified_task5')
TASK7B_SOURCE=None
for candidate in candidates7b:
    try:
        verified_task7b_trace_bytes(candidate);TASK7B_SOURCE=candidate.resolve();break
    except (ValueError,FileNotFoundError,IsADirectoryError,PermissionError,RuntimeError):pass
assert TASK7B_SOURCE is not None,'Aggiungi giada_cahva_active_closed_loop_microcanary.zip agli Input oppure imposta GIADA_TASK7B_ARTIFACT.'
print({'task5':str(TASK5_SOURCE),'task7b':str(TASK7B_SOURCE),'hashes_verified':True})


## 🧪 Ricalcolo dei bordi
La cella ricalcola 24 episodi usando soltanto tracce teacher già salvate. Formula, LUT e tre seed physical-tau rimangono congelati. Un sottoprocesso protegge il kernel.

In [ ]:
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_cahva_boundary_semantics_reassessment')
assert not OUTPUT_DIR.exists() or not any(p.name!='.verified_task5' for p in OUTPUT_DIR.iterdir()),f'Output con risultati già presenti: {OUTPUT_DIR}'
MOD=TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod'
command=[sys.executable,'-u',str(GIADA_REPO/'scripts/run_cahva_boundary_semantics_reassessment.py'),'--mod',str(MOD),'--task5-source',str(TASK5_ROOT),'--task7b-source',str(TASK7B_SOURCE),'--output-dir',str(OUTPUT_DIR),'--revision',str(REVISION)]
completed=subprocess.run(command,check=False)
if completed.returncode:raise RuntimeError(f'Task 7c fallita nel sottoprocesso (exit={completed.returncode}); conserva le ultime righe del log.')
report=json.loads((OUTPUT_DIR/'final_report.json').read_text())
display({'valid':report['valid'],'episode_count':report['episode_count'],'old_lag_max_abs':report['old_formula_one_step_gate_lag_max_abs'],'corrected_formula_V_max_abs_mv':report['corrected_formula_voltage_max_abs_mv'],'corrected_formula_gate_max_rmse':report['corrected_formula_gate_max_rmse'],'corrected_formula_ica_max_abs':report['corrected_formula_current_max_abs_ma_cm2'],'teacher_regenerated':report['teacher_regenerated'],'model_retrained':report['model_retrained']})


## 📦 Scarica il report e le tracce corrette
Metodo Blob/base64 consueto del progetto.

In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_cahva_boundary_semantics_reassessment','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
